# Road Risk Intelligence — GPU Training, National KDE & Cloud Rendering

**`notebooks/train_gpu.ipynb` — Layer 3 of the 3-layer research workflow.**

This notebook is the *compute* layer of a deliberate division of labour. It does the
things that only a GPU-backed cloud runtime can do well, and it refuses to do the things
that belong somewhere else.

```
┌──────────────────────────┐   ┌──────────────────────────┐   ┌──────────────────────────┐
│  LAYER 1 — RESEARCH      │   │  LAYER 2 — ENGINEERING   │   │  LAYER 3 — COMPUTE       │
│  Research notebook       │──▶│  Local IDE + pytest      │──▶│  Colab (this notebook)   │
│                          │   │                          │   │                          │
│ • literature sweep       │   │ • typed algorithms in    │   │ • GPU training at scale  │
│ • hypothesis register    │   │   src/road_risk_modeling │   │ • national spatial KDE   │
│ • metric + baseline      │   │ • pytest unit tests      │   │ • Folium / Kepler.gl     │
│   selection              │   │ • deterministic CPU      │   │   cloud rendering        │
│ • data-source appraisal  │   │   reference impl.        │   │ • artifact export        │
└──────────────────────────┘   └──────────────────────────┘   └──────────────────────────┘
          research.md                  git commit                  data/reporting/st_gnn
```

### Why this shape

The failure mode of notebook-first deep-learning research is that the *algorithm* ends up
living in a notebook: untested, unversioned, unreviewable, and impossible to run in the
product. Here the algorithm's contract lives in the repository
(`src/road_risk_modeling/st_gnn_forecasting.py`), covered by unit tests. This notebook
*imports* that contract, trains a much larger parameterisation of it on a GPU, and writes
artifacts back in **exactly** the schema the repository's CLI and API already consume.

If this notebook and the repository ever disagree about a schema, the repository wins.

---

### What this notebook produces

| Artifact | Path | Consumed by |
|---|---|---|
| Segment risk forecasts | `data/reporting/st_gnn/<label>_st_gnn_forecasts.csv` | `road-risk st-gnn-forecast`, `apps/api` |
| High-risk corridor alerts | `data/reporting/st_gnn/<label>_high_risk_corridor_alerts.csv` | operations board |
| JSON bundle + Markdown bulletin + manifest | same directory | `apps/web` reports hub |
| Trained weights + optimiser state | `checkpoints/` (Drive) | resumption, further fine-tuning |
| Model card | `checkpoints/model_card.json` | `src/road_risk_modeling/model_registry.py` |
| National KDE raster + per-segment density | `data/reporting/kde/` | siting advisor, hotspot screening |
| Interactive maps | `renders/*.html` | stakeholder review, Horizon Europe deliverables |

## Layer contract — read this before changing anything

**Layer 1 (research notebook) hands over:** a research memo naming the target metric, the
baseline to beat, the horizon set, the data sources with their licences, and the failure
modes to watch. No code.

**Layer 2 (local IDE) hands over:** committed, typed, unit-tested Python in
`src/road_risk_modeling/`. In particular this notebook depends on:

- `st_gnn_forecasting.RoadGraphNode`, `GraphEdge`, `SpatioTemporalFeatureFrame`
- `st_gnn_forecasting.SegmentRiskForecast`, `HighRiskCorridorAlert`
- `st_gnn_forecasting.build_network_graph_adjacency` — the *reference* CPU adjacency
- `st_gnn_forecasting.export_st_gnn_artifacts` — the *only* sanctioned writer

**Layer 3 (this notebook) must not:**

1. Redefine any dataclass that Layer 2 owns — import it.
2. Hand-roll artifact writers — call `export_st_gnn_artifacts`.
3. Become the home of an algorithm. If a cell here grows a non-trivial, reusable
   function, it belongs in `src/` with tests. There is a handoff checklist at the end.
4. Depend on hidden state from a previous run. Every cell is restart-safe.

**Anything trained here is a hypothesis until the ablations and calibration checks below
pass.** Unvalidated weights do not go back to Layer 2.

## 1. Runtime preflight

Run this first. It refuses to waste a GPU session on a misconfigured runtime, and it
records the exact hardware into the model card so results are attributable.

> **Colab:** Runtime → Change runtime type → **T4 / L4 / A100**. An A100 or L4 will run the
> national-scale configuration comfortably; a T4 needs `Config.batch_size` halved and
> `Config.amp_dtype = "float16"`.

In [ ]:
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules or os.environ.get("COLAB_RELEASE_TAG") is not None

print(f"python           : {platform.python_version()}")
print(f"platform         : {platform.platform()}")
print(f"in colab         : {IN_COLAB}")

try:
    import torch
except ModuleNotFoundError:  # pragma: no cover - local runtimes without torch
    torch = None
    print("torch            : NOT INSTALLED (the install cell below will handle it)")

if torch is not None:
    print(f"torch            : {torch.__version__}")
    print(f"cuda available   : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print(f"gpu              : {props.name}")
        print(f"vram             : {props.total_memory / 1024 ** 3:.1f} GiB")
        print(f"compute cap      : {props.major}.{props.minor}")
        print(f"bf16 supported   : {torch.cuda.is_bf16_supported()}")
    else:
        print("!! No GPU. Training cells will run but will be ~50x slower.")
        print("!! Colab: Runtime -> Change runtime type -> T4/L4/A100.")

try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False).stdout.strip()[:1200])
except FileNotFoundError:
    print("nvidia-smi not present on this runtime.")

In [ ]:
# Host resources. National-scale KDE is RAM-bound before it is VRAM-bound.
import shutil

try:
    import psutil

    vm = psutil.virtual_memory()
    print(f"host ram         : {vm.total / 1024 ** 3:.1f} GiB total, {vm.available / 1024 ** 3:.1f} GiB available")
except ModuleNotFoundError:
    print("psutil not available; skipping RAM report")

usage = shutil.disk_usage("/content" if IN_COLAB else ".")
print(f"disk             : {usage.free / 1024 ** 3:.1f} GiB free of {usage.total / 1024 ** 3:.1f} GiB")
print()
print("Rule of thumb for this notebook:")
print("  - 250k segments x 168 timesteps x 10 features (float32) ~= 1.7 GiB per full tensor")
print("  - the loader below keeps the panel on CPU and moves windows to GPU per batch")

## 2. Dependencies

Colab already ships `torch`, `numpy`, `pandas`, `scipy`, `matplotlib` and `folium`. Only
the genuinely missing pieces are installed, and versions are pinned so a rerun six months
from now reproduces today's numbers.

`torch-geometric` is deliberately **not** used: the diffusion convolution here is a sparse
matrix product, which `torch.sparse` does natively. One less brittle dependency, and the
maths stays visible and auditable.

In [ ]:
REQUIREMENTS = [
    "keplergl==0.3.2",      # interactive WebGL rendering
    "geopandas==1.0.1",     # segment geometry handling
    "pyproj==3.7.0",        # metric CRS reprojection for KDE
    "shapely==2.0.6",
    "scikit-learn==1.5.2",  # PR-AUC, calibration curves
    "psutil",
]

if IN_COLAB:
    import subprocess

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *REQUIREMENTS], check=True)
    print("installed:", ", ".join(REQUIREMENTS))
else:
    print("Not on Colab — install these into your own environment if missing:")
    print("  pip install " + " ".join(REQUIREMENTS))

## 3. Attach the repository (the Layer 2 source of truth)

Three supported modes, in order of preference:

1. **`clone`** — public clone of a pinned commit. Reproducible, no credentials.
2. **`drive`** — a working copy synced to Google Drive. Best for iterating with
   local development: edit locally, sync, re-run here.
3. **`local`** — you are running this notebook inside the repo (e.g. Jupyter on a
   workstation with a GPU).

Pin `REPO_REF` to a commit SHA for any run whose numbers you intend to publish. `main`
moves; a SHA does not.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/KOKUTBON/xROAD.git"
REPO_REF = "main"          # pin to a commit SHA for publishable runs
ATTACH_MODE = "clone"      # "clone" | "drive" | "local"
DRIVE_REPO_PATH = "/content/drive/MyDrive/ROAD-ACCIDENTS-RISK-MODELING"

if ATTACH_MODE == "drive":
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_ROOT = Path(DRIVE_REPO_PATH)
elif ATTACH_MODE == "clone":
    REPO_ROOT = Path("/content/ROAD-ACCIDENTS-RISK-MODELING" if IN_COLAB else "./ROAD-ACCIDENTS-RISK-MODELING")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "--filter=blob:none", REPO_URL, str(REPO_ROOT)], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "--all", "--quiet"], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--quiet", REPO_REF], check=True)
else:
    REPO_ROOT = Path.cwd()
    while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").exists():
        REPO_ROOT = REPO_ROOT.parent

assert (REPO_ROOT / "pyproject.toml").exists(), f"No repository at {REPO_ROOT}"

GIT_SHA = subprocess.run(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
    capture_output=True, text=True, check=False,
).stdout.strip() or "unknown"

print(f"repo root : {REPO_ROOT}")
print(f"git sha   : {GIT_SHA}")

In [ ]:
# Editable install so `import road_risk_modeling` resolves to the checked-out tree and
# any local edit is picked up on the next kernel restart.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_ROOT}[analytics,geospatial,ml,dev]"],
    check=True,
)

if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from road_risk_modeling.st_gnn_forecasting import (  # noqa: E402
    DEFAULT_HORIZONS,
    ST_GNN_CONTRACT_VERSION,
    ST_GNN_ENGINE_VERSION,
    GraphEdge,
    HighRiskCorridorAlert,
    RoadGraphNode,
    SegmentRiskForecast,
    SpatioTemporalFeatureFrame,
    build_network_graph_adjacency,
    execute_st_gnn_forecast,
    export_st_gnn_artifacts,
)

print(f"engine version   : {ST_GNN_ENGINE_VERSION}")
print(f"contract version : {ST_GNN_CONTRACT_VERSION}")
print(f"default horizons : {DEFAULT_HORIZONS}")

### 3.1 Gate the GPU spend on Layer 2's tests

A GPU session is a scarce, billed resource. Before committing to it, prove the engineering
layer is intact on this exact commit. If these tests fail, stop: the contract this
notebook writes against is broken, and any artifact produced would be unusable downstream.

In [ ]:
# PYTHONPATH is set explicitly: the subprocess does not inherit this notebook's
# sys.path, so without it the tests cannot import the package under ATTACH_MODE="local".
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q",
     str(REPO_ROOT / "tests" / "test_st_gnn_forecasting.py"),
     "--no-header", "-x"],
    capture_output=True, text=True, check=False, cwd=str(REPO_ROOT),
    env={**os.environ, "PYTHONPATH": str(REPO_ROOT / "src")},
)
print(test_result.stdout[-2500:])
print(test_result.stderr[-800:])

CONTRACT_OK = test_result.returncode == 0
if not CONTRACT_OK:
    print("\n!! Layer 2 contract tests did not pass on this commit.")
    print("!! Fix them locally and re-pin REPO_REF before training.")
else:
    print("\nContract tests pass — safe to spend GPU time.")

## 4. Reproducibility

Three separate things get confused under the word "reproducible":

- **Seeded** — the same seed gives the same sample order and initialisation.
- **Deterministic** — the same seed *and* the same hardware give bit-identical results.
  cuDNN's autotuner and some scatter kernels are non-deterministic by default; forcing
  determinism costs roughly 5–15% throughput.
- **Attributable** — the run records its commit, hardware, library versions and config
  hash, so a number in a paper can be traced to the run that produced it.

Research runs should be attributable and seeded. Only turn on strict determinism when
chasing a discrepancy, and say so in the model card.

In [ ]:
import hashlib
import math
import random
import time
from datetime import UTC, datetime

import numpy as np
import torch

SEED = 20260917
STRICT_DETERMINISM = False


def seed_everything(seed: int, strict: bool = False) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if strict:
        os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
        torch.use_deterministic_algorithms(True, warn_only=True)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    else:
        torch.backends.cudnn.benchmark = True


seed_everything(SEED, STRICT_DETERMINISM)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# TF32 keeps fp32 matmuls fast on Ampere+ with no meaningful accuracy cost here.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

RUN_ID = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
print(f"device : {DEVICE}")
print(f"seed   : {SEED} (strict determinism: {STRICT_DETERMINISM})")
print(f"run id : {RUN_ID}")

## 5. Configuration

One frozen dataclass holds every knob. It is hashed into the model card, so two runs with
the same hash are the same experiment. Change a value here, never inline further down —
inline overrides are how results stop being reproducible.

In [ ]:
from dataclasses import asdict, dataclass, field


@dataclass(frozen=True)
class TrainConfig:
    # --- network scale ---------------------------------------------------
    n_segments: int = 12_000          # raise to 150_000+ for a full national network
    history_hours: int = 48           # T: timesteps fed to the temporal stack
    horizons: tuple[int, ...] = (1, 6, 24)
    panel_hours: int = 24 * 70        # total simulated/loaded hours of history

    # --- graph -----------------------------------------------------------
    knn_neighbours: int = 8           # sparse adjacency degree
    cluster_size: int = 2_048         # nodes per training subgraph (Cluster-GCN batching)
    gaussian_sigma_m: float = 1_000.0     # mirrors build_network_graph_adjacency
    max_neighbor_distance_m: float = 2_500.0
    diffusion_steps: int = 2          # K in the diffusion convolution

    # --- model -----------------------------------------------------------
    in_features: int = 10
    hidden_channels: int = 48
    skip_channels: int = 128
    end_channels: int = 256
    n_blocks: int = 4                 # each block doubles the dilation
    kernel_size: int = 2
    adaptive_embedding_dim: int = 16  # learned adjacency (Graph WaveNet's A_adp)
    dropout: float = 0.15

    # --- optimisation ----------------------------------------------------
    batch_size: int = 8               # halve on a T4
    grad_accumulation: int = 1
    epochs: int = 30
    lr: float = 2e-3
    weight_decay: float = 1e-4
    grad_clip_norm: float = 3.0
    warmup_frac: float = 0.05
    early_stop_patience: int = 6
    amp_dtype: str = "bfloat16"       # "bfloat16" on A100/L4, "float16" on T4
    use_torch_compile: bool = False   # first-call compile cost is ~2 min

    # --- loss ------------------------------------------------------------
    focal_gamma: float = 2.0
    focal_alpha: float = 0.75         # crash-positive hours are rare
    horizon_weights: tuple[float, ...] = (1.0, 0.8, 0.6)

    # --- splits (temporal, never random) ---------------------------------
    val_frac: float = 0.15
    test_frac: float = 0.15

    # --- KDE -------------------------------------------------------------
    kde_bandwidth_m: float = 350.0
    kde_cell_m: float = 100.0
    kde_truncation_sigmas: float = 3.0

    # --- labels ----------------------------------------------------------
    network_label: str = "national_network"
    alert_threshold: float = 0.55

    def hash(self) -> str:
        payload = json.dumps(asdict(self), sort_keys=True, default=str).encode("utf-8")
        return hashlib.sha256(payload).hexdigest()[:16]


import json  # noqa: E402  (used by TrainConfig.hash)

CFG = TrainConfig()
assert len(CFG.horizon_weights) == len(CFG.horizons), "one weight per horizon"
assert CFG.history_hours > max(CFG.horizons), "history must exceed the longest horizon"

print(json.dumps(asdict(CFG), indent=2, default=str))
print(f"\nconfig hash : {CFG.hash()}")

In [ ]:
# Output locations. Everything the notebook writes lands under these four roots.
ARTIFACT_ROOT = Path("/content/artifacts" if IN_COLAB else "./artifacts") / RUN_ID
REPORT_DIR = ARTIFACT_ROOT / "reporting" / "st_gnn"
KDE_DIR = ARTIFACT_ROOT / "reporting" / "kde"
CKPT_DIR = ARTIFACT_ROOT / "checkpoints"
RENDER_DIR = ARTIFACT_ROOT / "renders"

for directory in (REPORT_DIR, KDE_DIR, CKPT_DIR, RENDER_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"artifacts -> {ARTIFACT_ROOT}")

## 6. Data layer

### 6.1 What a national network looks like to this model

| Object | Shape | Meaning |
|---|---|---|
| Nodes | `N` | road segments, each with length, free-flow speed, functional class, centroid |
| Edges | `E ≈ N·k` | sparse topological + proximity adjacency |
| Panel | `(H, N, F)` | `H` hourly timesteps × `N` segments × `F` dynamic features |
| Labels | `(H, N, |horizons|)` | did a crash occur on this segment within each lookahead |

At national scale a dense `N × N` adjacency is impossible — 150k segments would need 90 GB
in fp32. The repository's reference implementation
(`build_network_graph_adjacency`) is dense on purpose: it is a readable, testable
specification at city scale. Here we build the **sparse equivalent** and assert the two
agree on a small subgraph, so the GPU path stays faithful to the tested one.

### 6.2 Loading real data

Point `GRAPH_JSON` at a graph file in the schema the CLI already reads
(`road-risk st-gnn-forecast --graph-path`):

```json
{
  "nodes": [{"segment_id": "...", "length_km": 0.4, "free_flow_speed_kmh": 50,
             "functional_class": "urban_arterial", "latitude": 41.1, "longitude": 16.8}],
  "edges": [{"from_segment_id": "...", "to_segment_id": "...", "distance_m": 120.0}],
  "time_series_frames": [{"segment_id": "...", "timestamp": "...", "...": "..."}]
}
```

If no file is present the cell synthesises a **deterministic** national network with
realistic structure — radial corridors around population centres, functional-class speed
profiles, diurnal and weekly demand cycles, weather shocks, and crash counts drawn from a
negative-binomial process whose rate depends on those covariates. It is a stand-in for
plumbing and scaling work, **not** a source of publishable safety findings. The banner it
prints is there to stop synthetic numbers ever being reported as real ones.

In [ ]:
GRAPH_JSON = REPO_ROOT / "data" / "planning" / "national_hierarchy" / "national_graph.json"

FUNCTIONAL_CLASSES = ("motorway", "trunk", "urban_arterial", "urban_collector", "rural_local")
CLASS_SPEEDS = {"motorway": 120.0, "trunk": 90.0, "urban_arterial": 50.0,
                "urban_collector": 40.0, "rural_local": 60.0}
CLASS_BASE_RISK = {"motorway": 0.35, "trunk": 0.45, "urban_arterial": 1.0,
                   "urban_collector": 0.7, "rural_local": 0.55}

IS_SYNTHETIC = not GRAPH_JSON.exists()


def synthesise_national_network(cfg: TrainConfig, rng: np.random.Generator) -> dict:
    """Deterministic national-scale road network with plausible spatial structure."""
    n = cfg.n_segments
    n_centres = max(6, n // 3_000)

    # Population centres scattered over a country-sized bounding box (approx. Italy).
    centre_lat = rng.uniform(37.0, 46.5, size=n_centres)
    centre_lon = rng.uniform(7.0, 18.0, size=n_centres)
    centre_weight = rng.dirichlet(np.ones(n_centres) * 1.4)

    assignment = rng.choice(n_centres, size=n, p=centre_weight)
    # Radial decay away from each centre: dense urban core, sparse rural periphery.
    radius_deg = np.abs(rng.normal(0.0, 0.22, size=n)) ** 1.5
    bearing = rng.uniform(0.0, 2.0 * np.pi, size=n)
    lat = centre_lat[assignment] + radius_deg * np.cos(bearing)
    lon = centre_lon[assignment] + radius_deg * np.sin(bearing) * 1.3

    urbanity = np.exp(-radius_deg / 0.18)  # 1.0 at a centre, ~0 in the countryside
    class_probs = np.stack([
        0.04 + 0.02 * (1 - urbanity),
        0.08 + 0.04 * (1 - urbanity),
        0.30 * urbanity + 0.05,
        0.34 * urbanity + 0.05,
        0.55 * (1 - urbanity) + 0.05,
    ], axis=1)
    class_probs /= class_probs.sum(axis=1, keepdims=True)
    class_idx = np.array([rng.choice(5, p=row) for row in class_probs])
    classes = np.array(FUNCTIONAL_CLASSES)[class_idx]

    length_km = np.clip(rng.gamma(shape=2.0, scale=0.22, size=n) * (2.0 - urbanity), 0.05, 12.0)
    free_flow = np.array([CLASS_SPEEDS[c] for c in classes])
    aadt = np.clip(rng.lognormal(mean=8.6, sigma=0.9, size=n) * (0.4 + 1.6 * urbanity), 120, 180_000)

    return {
        "segment_id": np.array([f"seg-{i:07d}" for i in range(n)]),
        "latitude": lat,
        "longitude": lon,
        "length_km": length_km,
        "free_flow_speed_kmh": free_flow,
        "functional_class": classes,
        "aadt": aadt,
        "urbanity": urbanity,
    }


rng = np.random.default_rng(SEED)

if IS_SYNTHETIC:
    print("=" * 78)
    print("SYNTHETIC NETWORK IN USE — plumbing and scaling only.")
    print("Numbers produced from this network are NOT road-safety findings and must")
    print("never be published or fed to the reporting hub as if they were.")
    print("=" * 78)
    NET = synthesise_national_network(CFG, rng)
else:
    raw = json.loads(GRAPH_JSON.read_text(encoding="utf-8"))
    nodes_raw = raw["nodes"]
    NET = {
        "segment_id": np.array([str(nd["segment_id"]) for nd in nodes_raw]),
        "latitude": np.array([float(nd["latitude"]) for nd in nodes_raw]),
        "longitude": np.array([float(nd["longitude"]) for nd in nodes_raw]),
        "length_km": np.array([float(nd.get("length_km", 0.5)) for nd in nodes_raw]),
        "free_flow_speed_kmh": np.array([float(nd.get("free_flow_speed_kmh", 50.0)) for nd in nodes_raw]),
        "functional_class": np.array([str(nd.get("functional_class", "urban_arterial")) for nd in nodes_raw]),
        "aadt": np.array([float(nd.get("aadt", 8_000.0)) for nd in nodes_raw]),
        "urbanity": np.array([float(nd.get("urbanity", 0.5)) for nd in nodes_raw]),
    }
    print(f"Loaded {len(NET['segment_id']):,} real segments from {GRAPH_JSON}")

N_SEGMENTS = len(NET["segment_id"])
print(f"segments : {N_SEGMENTS:,}")
print(f"bbox     : lat [{NET['latitude'].min():.2f}, {NET['latitude'].max():.2f}], "
      f"lon [{NET['longitude'].min():.2f}, {NET['longitude'].max():.2f}]")

### 6.3 Metric projection

Everything spatial — adjacency radii, KDE bandwidths, density per km² — must be computed
in metres, not degrees. A degree of longitude is 111 km at the equator and 78 km at 45°N;
doing kernel density in degrees silently stretches every hotspot east-west.

We project to a local azimuthal equidistant CRS centred on the network, which keeps
distances accurate across a whole country far better than a single UTM zone would.

In [ ]:
from pyproj import CRS, Transformer

LAT0 = float(np.median(NET["latitude"]))
LON0 = float(np.median(NET["longitude"]))
METRIC_CRS = CRS.from_proj4(
    f"+proj=aeqd +lat_0={LAT0} +lon_0={LON0} +x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"
)
TO_METRIC = Transformer.from_crs("EPSG:4326", METRIC_CRS, always_xy=True)
TO_WGS84 = Transformer.from_crs(METRIC_CRS, "EPSG:4326", always_xy=True)

X_M, Y_M = TO_METRIC.transform(NET["longitude"], NET["latitude"])
XY_M = np.column_stack([np.asarray(X_M, dtype=np.float64), np.asarray(Y_M, dtype=np.float64)])

print(f"projected extent : {(XY_M[:, 0].max() - XY_M[:, 0].min()) / 1000:,.0f} km east-west, "
      f"{(XY_M[:, 1].max() - XY_M[:, 1].min()) / 1000:,.0f} km north-south")

### 6.4 Sparse graph construction

`build_network_graph_adjacency` (Layer 2) builds a dense Gaussian-kernel adjacency with a
hard distance cutoff, then row-normalises it into a transition matrix `P`. We reproduce
exactly that weighting, but keep only each node's `k` nearest neighbours inside the cutoff
— which is where essentially all of the mass lives anyway, since the Gaussian has already
decayed to `exp(-(2500/1000)²) ≈ 0.002` at the cutoff.

The cell then **verifies** the sparse path against the dense reference on a random
subgraph. This assertion is the whole reason the GPU path can be trusted.

In [ ]:
from scipy.spatial import cKDTree

tree = cKDTree(XY_M)
# k+1 because the first neighbour returned is the node itself.
dists, idxs = tree.query(XY_M, k=min(CFG.knn_neighbours + 1, N_SEGMENTS), workers=-1)

src_list, dst_list, w_list = [], [], []
sigma2 = CFG.gaussian_sigma_m ** 2
for col in range(1, idxs.shape[1]):
    d = dists[:, col]
    keep = d <= CFG.max_neighbor_distance_m
    src_list.append(np.nonzero(keep)[0])
    dst_list.append(idxs[keep, col])
    w_list.append(np.exp(-(d[keep] ** 2) / sigma2))

src = np.concatenate(src_list)
dst = np.concatenate(dst_list)
wgt = np.concatenate(w_list)

# Row-normalise into a transition matrix, exactly as the reference does.
row_sum = np.zeros(N_SEGMENTS, dtype=np.float64)
np.add.at(row_sum, src, wgt)
row_sum[row_sum == 0.0] = 1.0
wgt_norm = wgt / row_sum[src]

def _row_normalise_np(w: np.ndarray) -> np.ndarray:
    """Row-stochastic transition matrix with the reference engine's self-loop rule.

    A node with no neighbour inside the cutoff keeps P[i][i] = 1.0 rather than a zero
    row — otherwise the diffusion step erases that segment's own signal entirely, which
    silently drops isolated rural segments from the model.
    """
    p = np.array(w, dtype=np.float64, copy=True)
    row_sum = p.sum(axis=1)
    isolated = row_sum <= 1e-6
    p = p / np.where(isolated, 1.0, row_sum)[:, None]
    idx = np.nonzero(isolated)[0]
    p[idx, idx] = 1.0
    return p


EDGE_INDEX = torch.from_numpy(np.stack([src, dst])).long()
EDGE_WEIGHT = torch.from_numpy(wgt_norm).float()

density = len(src) / float(N_SEGMENTS * N_SEGMENTS)
print(f"edges    : {len(src):,}  (mean degree {len(src) / N_SEGMENTS:.2f})")
print(f"density  : {density:.2e}  — a dense matrix would need "
      f"{N_SEGMENTS ** 2 * 4 / 1024 ** 3:,.1f} GiB")

In [ ]:
# --- fidelity check against the Layer 2 reference implementation ----------
# The subgraph must be spatially LOCAL. Sampling 48 nodes at random from a national
# network gives a set whose pairwise distances all exceed the 2.5 km cutoff, so both
# matrices collapse to self-loops and the comparison tests nothing.
seed_node = int(rng.integers(0, N_SEGMENTS))
_, local_ids = tree.query(XY_M[seed_node], k=min(48, N_SEGMENTS))
sub_ids = np.sort(np.atleast_1d(local_ids))

sub_nodes = [
    RoadGraphNode(
        segment_id=str(NET["segment_id"][i]),
        length_km=float(NET["length_km"][i]),
        free_flow_speed_kmh=float(NET["free_flow_speed_kmh"][i]),
        functional_class=str(NET["functional_class"][i]),
        latitude=float(NET["latitude"][i]),
        longitude=float(NET["longitude"][i]),
    )
    for i in sub_ids
]

_ref_index, ref_P, _ref_adp = build_network_graph_adjacency(
    sub_nodes,
    edges=None,
    gaussian_sigma_m=CFG.gaussian_sigma_m,
    max_neighbor_distance_m=CFG.max_neighbor_distance_m,
)
ref_P = np.asarray(ref_P, dtype=np.float64)

# Same construction from our projected coordinates, including the reference's
# self-loop rule for isolated nodes.
sub_xy = XY_M[sub_ids]
d_ours = np.linalg.norm(sub_xy[:, None, :] - sub_xy[None, :, :], axis=-1)
w_sub = np.where(d_ours <= CFG.max_neighbor_distance_m, np.exp(-(d_ours ** 2) / sigma2), 0.0)
np.fill_diagonal(w_sub, 0.0)
ours_P = _row_normalise_np(w_sub)

row_corr = float(np.corrcoef(ref_P.ravel(), ours_P.ravel())[0, 1])
max_diff = float(np.abs(ref_P - ours_P).max())
print(f"local subgraph of {len(sub_ids)} nodes around segment {sub_ids[0]}")
print(f"reference vs notebook transition matrix, correlation : {row_corr:.4f}")
print(f"max |difference|                                     : {max_diff:.4f}")

# --- audit the distance convention ---------------------------------------
# The reference converts degrees to metres with fixed factors (85,000 m per degree of
# longitude, 111,000 per degree of latitude) — an equirectangular approximation exact
# at ~40 degrees N. Quantify the drift at THIS network's latitude rather than assuming
# it is negligible; the answer is a finding to hand back to Layer 2.
sub_ll = np.column_stack([NET["longitude"][sub_ids], NET["latitude"][sub_ids]])
d_ref = np.linalg.norm(
    (sub_ll[:, None, :] - sub_ll[None, :, :]) * np.array([85_000.0, 111_000.0]), axis=-1
)
in_range = (d_ours > 0) & (d_ours <= CFG.max_neighbor_distance_m)
if in_range.any():
    rel_err = np.abs(d_ref[in_range] - d_ours[in_range]) / d_ours[in_range]
    print(f"distance-convention drift at lat {LAT0:.1f}N: "
          f"median {np.median(rel_err) * 100:.2f}%, max {rel_err.max() * 100:.2f}%")
    if rel_err.max() > 0.05:
        print("  NOTE: >5% drift. The reference's fixed 85,000 m/degree factor is exact")
        print("        near 40N only. Worth raising as a Layer 2 issue for networks")
        print("        outside southern Europe.")

print()
if row_corr > 0.95:
    print("PASS — the notebook's adjacency matches the tested CPU reference.")
else:
    print("FAIL — investigate before training. The two layers disagree about the graph.")
    print("       Usual causes: a units mismatch (degrees vs metres), a missing self-loop")
    print("       rule for isolated nodes, or a subgraph that is not spatially local.")

### 6.5 Cluster batching — why the whole graph never enters the GPU at once

A full-graph forward pass is the thing that quietly makes ST-GNN code un-runnable at
national scale. The activation tensor is `(B, C, N, T)`: at `B=8`, `C=48`, `N=150,000`,
`T=48` that is 2.6 × 10⁹ floats — **10 GiB for a single layer's activations**, before
autograd saves anything for the backward pass, and the diffusion convolution concatenates
six of them.

The standard remedy is **Cluster-GCN batching**: partition the nodes into compact
subgraphs, and train on one subgraph at a time. For a road network the partition should be
*spatial*, because that is where the edges are — k-means on the projected coordinates keeps
the overwhelming majority of edges inside a cluster, so very little of the graph structure
is lost per step.

Two consequences worth stating plainly:

- **Each cluster's adjacency is small enough to be dense** (`2048 × 2048` ≈ 16 MiB), so the
  diffusion convolution becomes an ordinary batched matmul — faster and better-conditioned
  than sparse kernels, and far easier to read.
- **Cross-cluster edges are dropped within a step.** That is Cluster-GCN's known
  trade-off. The retention figure printed below is the number to watch: above ~90%, the
  loss is negligible against the memory saved. Long-range coupling that genuinely matters
  is still recoverable through the learned adaptive adjacency, and through the full-graph
  contagion pass used at inference time.

In [ ]:
from sklearn.cluster import MiniBatchKMeans

n_clusters = max(1, int(math.ceil(N_SEGMENTS / CFG.cluster_size)))
kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=SEED, n_init=3, batch_size=4096)
cluster_of = kmeans.fit_predict(XY_M)

NODE_BLOCKS = [np.nonzero(cluster_of == c)[0] for c in range(n_clusters)]
NODE_BLOCKS = [b for b in NODE_BLOCKS if len(b) >= 16]

intra = float(np.mean(cluster_of[src] == cluster_of[dst]))
print(f"clusters        : {len(NODE_BLOCKS)}  "
      f"(sizes {min(len(b) for b in NODE_BLOCKS)}-{max(len(b) for b in NODE_BLOCKS)})")
print(f"edge retention  : {intra * 100:.1f}% of edges stay inside a cluster")
if intra < 0.85:
    print("!! Low retention — raise cluster_size, or the spatial partition is fighting the graph.")


def dense_block_adjacency(block: np.ndarray) -> tuple[torch.Tensor, torch.Tensor]:
    """Row-normalised forward/backward transition matrices restricted to one cluster."""
    xy = XY_M[block]
    d = np.linalg.norm(xy[:, None, :] - xy[None, :, :], axis=-1)
    w = np.where(d <= CFG.max_neighbor_distance_m, np.exp(-(d ** 2) / sigma2), 0.0)
    np.fill_diagonal(w, 0.0)

    # Same self-loop rule as the reference engine, verified in the previous cell.
    fwd = torch.from_numpy(_row_normalise_np(w).astype(np.float32))
    bwd = torch.from_numpy(_row_normalise_np(w.T).astype(np.float32))
    return fwd, bwd


# Adjacencies are built once and cached on the GPU: n_clusters x (2048^2) x 4B x 2 is small.
BLOCK_ADJ = []
for block in NODE_BLOCKS:
    fwd, bwd = dense_block_adjacency(block)
    BLOCK_ADJ.append((fwd.to(DEVICE), bwd.to(DEVICE)))

adj_bytes = sum(f.numel() + b.numel() for f, b in BLOCK_ADJ) * 4
print(f"cached adjacency: {adj_bytes / 1024 ** 2:,.0f} MiB on {DEVICE}")

max_block = max(len(b) for b in NODE_BLOCKS)
act_bytes = CFG.batch_size * CFG.hidden_channels * max_block * CFG.history_hours * 4
print(f"peak activation : ~{act_bytes / 1024 ** 2:,.0f} MiB per layer "
      f"(x6 diffusion supports, halved again under bf16)")

### 6.6 The spatio-temporal panel

Ten dynamic features per segment per hour, matching `SpatioTemporalFeatureFrame` so the
exported forecasts stay interpretable against the CLI contract:

| # | Feature | Source in production |
|---|---|---|
| 0 | historical crash count (rolling) | crash register |
| 1 | telematics near-miss rate | `telematics_engine` |
| 2 | speed ratio (current / free-flow) | floating-car / loop detectors |
| 3 | precipitation mm/h | `weather_ingestion` |
| 4 | wet-or-icy surface flag | `weather_risk_model` |
| 5 | visibility (normalised, inverted) | weather |
| 6 | hour-of-day sine | calendar |
| 7 | hour-of-day cosine | calendar |
| 8 | peak-hour flag | calendar |
| 9 | log AADT exposure (static, broadcast) | exposure connectors |

**Labels** are binary crash occurrence within each horizon. The positive rate on a real
national network is on the order of 1e-4 per segment-hour; the focal loss and the
threshold sweep below exist because of that, not as decoration.

In [ ]:
def build_panel(net: dict, cfg: TrainConfig, rng: np.random.Generator):
    """Return X (H, N, F) float32 and Y (H, N, n_horizons) float32."""
    n = len(net["segment_id"])
    hours = cfg.panel_hours
    hour_of_day = np.arange(hours) % 24
    day_of_week = (np.arange(hours) // 24) % 7

    # Demand: diurnal double peak, weekday/weekend contrast.
    diurnal = (
        0.55
        + 0.45 * np.exp(-((hour_of_day - 8) ** 2) / 6.0)
        + 0.50 * np.exp(-((hour_of_day - 18) ** 2) / 8.0)
        - 0.35 * np.exp(-((hour_of_day - 3) ** 2) / 6.0)
    )
    weekend = np.isin(day_of_week, (5, 6))
    diurnal = diurnal * np.where(weekend, 0.78, 1.0)

    # Weather: spatially correlated fronts drifting across the country.
    front_phase = rng.uniform(0, 2 * np.pi, size=6)
    front_speed = rng.uniform(0.02, 0.08, size=6)
    xnorm = (net["latitude"] - net["latitude"].min()) / (np.ptp(net["latitude"]) + 1e-9)
    precip = np.zeros((hours, n), dtype=np.float32)
    for f in range(6):
        wave = np.sin(2 * np.pi * (front_speed[f] * np.arange(hours)[:, None] - xnorm[None, :]) + front_phase[f])
        precip += np.clip(wave - 0.72, 0.0, None) * rng.uniform(3.0, 14.0)
    precip = precip.astype(np.float32)

    wet = (precip > 0.25).astype(np.float32)
    visibility = np.clip(10_000.0 - precip * 520.0, 400.0, 10_000.0).astype(np.float32)

    urbanity = net["urbanity"].astype(np.float32)
    speed_ratio = np.clip(
        1.0
        - 0.34 * diurnal[:, None] * urbanity[None, :]
        - 0.06 * np.clip(precip / 8.0, 0.0, 1.0)
        + rng.normal(0.0, 0.03, size=(hours, n)),
        0.18, 1.0,
    ).astype(np.float32)

    log_aadt = np.log1p(net["aadt"]).astype(np.float32)
    log_aadt_z = ((log_aadt - log_aadt.mean()) / (log_aadt.std() + 1e-9)).astype(np.float32)
    class_risk = np.array([CLASS_BASE_RISK[c] for c in net["functional_class"]], dtype=np.float32)

    # Crash intensity: exposure x class x congestion x weather, Poisson-sampled.
    lam = (
        4.0e-4
        * np.exp(0.55 * log_aadt_z)[None, :]
        * class_risk[None, :]
        * (0.55 + 0.9 * diurnal[:, None])
        * (1.0 + 1.5 * (1.0 - speed_ratio))
        * (1.0 + 0.9 * wet + 0.05 * precip)
        * net["length_km"].astype(np.float32)[None, :]
    )
    crashes = rng.poisson(np.clip(lam, 0.0, 0.9)).astype(np.float32)

    # Near-miss telematics: a leading indicator, correlated with but noisier than crashes.
    near_miss = (
        38.0 * (1.0 - speed_ratio)
        + 12.0 * wet
        + 90.0 * crashes
        + rng.gamma(2.0, 3.5, size=(hours, n))
    ).astype(np.float32)

    # Rolling 24h crash history (causal: strictly past).
    csum = np.cumsum(crashes, axis=0)
    hist = np.zeros_like(crashes)
    hist[1:] = csum[:-1]
    hist[25:] -= csum[:-25]

    peak = np.isin(hour_of_day, (7, 8, 9, 17, 18, 19)).astype(np.float32)

    x = np.empty((hours, n, cfg.in_features), dtype=np.float32)
    x[..., 0] = hist
    x[..., 1] = near_miss / 150.0
    x[..., 2] = speed_ratio
    x[..., 3] = precip / 15.0
    x[..., 4] = wet
    x[..., 5] = 1.0 - visibility / 10_000.0
    x[..., 6] = np.sin(2 * np.pi * hour_of_day / 24.0)[:, None]
    x[..., 7] = np.cos(2 * np.pi * hour_of_day / 24.0)[:, None]
    x[..., 8] = peak[:, None]
    x[..., 9] = log_aadt_z[None, :]

    # Labels: any crash within each lookahead window.
    y = np.zeros((hours, n, len(cfg.horizons)), dtype=np.float32)
    occurred = (crashes > 0).astype(np.float32)
    for hi, horizon in enumerate(cfg.horizons):
        fwd = np.zeros_like(occurred)
        for step in range(1, horizon + 1):
            fwd[:-step] = np.maximum(fwd[:-step], occurred[step:])
        y[..., hi] = fwd

    return x, y, crashes


X_PANEL, Y_PANEL, CRASH_COUNTS = build_panel(NET, CFG, rng)

print(f"X : {X_PANEL.shape}  ({X_PANEL.nbytes / 1024 ** 3:.2f} GiB)")
print(f"Y : {Y_PANEL.shape}")
for hi, horizon in enumerate(CFG.horizons):
    print(f"  positive rate @ {horizon:>2}h : {Y_PANEL[..., hi].mean():.5f}")

### 6.7 Temporal splits and windowing

**Never shuffle a spatio-temporal panel into random splits.** A randomly-held-out hour sits
between two hours the model has memorised, and the reported score becomes a measure of
interpolation rather than forecasting. Splits here are strictly chronological, with a gap
of `max(horizons)` hours between them so no label window straddles a boundary.

The windows are **rolling-origin**: a validation window's 48-hour context may reach back
across the boundary into training hours. That is not leakage — at prediction time the
recent past is exactly what you have. What must never cross is a *label*, and the gap is
what guarantees it.

Feature normalisation statistics come from the **training slice only**, then are applied
to validation and test. Fitting the scaler on everything is the most common silent leak in
this class of model.

In [ ]:
from torch.utils.data import DataLoader, Dataset

H_TOTAL = X_PANEL.shape[0]
GAP = max(CFG.horizons)
n_test = int(H_TOTAL * CFG.test_frac)
n_val = int(H_TOTAL * CFG.val_frac)
n_train = H_TOTAL - n_val - n_test

TRAIN_SLICE = slice(0, n_train)
VAL_SLICE = slice(n_train + GAP, n_train + n_val)
TEST_SLICE = slice(n_train + n_val + GAP, H_TOTAL)

# Scaler fitted on training hours only. Cyclical and flag features are left alone.
SCALE_COLS = [0, 1, 2, 3, 5, 9]
mu = X_PANEL[TRAIN_SLICE].reshape(-1, CFG.in_features).mean(axis=0)
sd = X_PANEL[TRAIN_SLICE].reshape(-1, CFG.in_features).std(axis=0) + 1e-6
mu_t = np.zeros(CFG.in_features, dtype=np.float32)
sd_t = np.ones(CFG.in_features, dtype=np.float32)
mu_t[SCALE_COLS] = mu[SCALE_COLS]
sd_t[SCALE_COLS] = sd[SCALE_COLS]
X_PANEL = (X_PANEL - mu_t) / sd_t


class PanelWindows(Dataset):
    """Sliding time windows crossed with spatial node blocks.

    One item is (history window, labels, block id) for a single cluster, which is what
    keeps the activation tensor bounded regardless of how large the national network is.
    A batch is drawn from one block at a time, so the whole batch shares an adjacency.
    """

    def __init__(self, x, y, span: slice, history: int, blocks: list):
        # Rolling-origin windows: the *context* may reach back across the split boundary
        # (that history is genuinely available at prediction time, so using it is not
        # leakage), but the *label* always falls inside this split, which is what the gap
        # between splits enforces.
        self.starts = np.arange(max(span.start, history), span.stop, dtype=np.int64)
        self.x = x
        self.y = y
        self.history = history
        self.blocks = blocks
        # Item index -> (time index, block index).
        self.pairs = [(t, b) for b in range(len(blocks)) for t in range(len(self.starts))]

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, i: int):
        ti, bi = self.pairs[i]
        t = int(self.starts[ti])
        block = self.blocks[bi]
        window = np.ascontiguousarray(self.x[t - self.history:t, block])   # (T, nb, F)
        target = np.ascontiguousarray(self.y[t - 1, block])                # (nb, n_horizons)
        return torch.from_numpy(window), torch.from_numpy(target), bi


class BlockBatchSampler(torch.utils.data.Sampler):
    """Yield batches whose items all come from the same node block."""

    def __init__(self, dataset: PanelWindows, batch_size: int, shuffle: bool, drop_last: bool):
        self.by_block: dict[int, list[int]] = {}
        for idx, (_t, b) in enumerate(dataset.pairs):
            self.by_block.setdefault(b, []).append(idx)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.drop_last = drop_last

    def __iter__(self):
        batches = []
        for indices in self.by_block.values():
            order = list(indices)
            if self.shuffle:
                random.shuffle(order)
            for start in range(0, len(order), self.batch_size):
                chunk = order[start:start + self.batch_size]
                if self.drop_last and len(chunk) < self.batch_size:
                    continue
                batches.append(chunk)
        if self.shuffle:
            random.shuffle(batches)
        return iter(batches)

    def __len__(self) -> int:
        return sum(
            len(v) // self.batch_size if self.drop_last else math.ceil(len(v) / self.batch_size)
            for v in self.by_block.values()
        )


def collate_block(items):
    windows = torch.stack([it[0] for it in items])
    targets = torch.stack([it[1] for it in items])
    return windows, targets, items[0][2]


# In-process loading. The panel already lives in RAM and a window is a slice, so worker
# processes buy little here — and they cost a lot: a worker must re-import and re-pickle
# every class defined in this notebook, which is exactly the kind of failure that bites
# an hour into a training run. `pin_memory` still gives asynchronous host-to-device
# copies, which is where the real overlap comes from.
LOADER_WORKERS = 0


def make_loader(ds: PanelWindows, shuffle: bool, drop_last: bool) -> DataLoader:
    return DataLoader(
        ds,
        batch_sampler=BlockBatchSampler(ds, CFG.batch_size, shuffle, drop_last),
        collate_fn=collate_block,
        num_workers=LOADER_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=LOADER_WORKERS > 0,
    )


train_ds = PanelWindows(X_PANEL, Y_PANEL, TRAIN_SLICE, CFG.history_hours, NODE_BLOCKS)
val_ds = PanelWindows(X_PANEL, Y_PANEL, VAL_SLICE, CFG.history_hours, NODE_BLOCKS)
test_ds = PanelWindows(X_PANEL, Y_PANEL, TEST_SLICE, CFG.history_hours, NODE_BLOCKS)

train_dl = make_loader(train_ds, shuffle=True, drop_last=True)
val_dl = make_loader(val_ds, shuffle=False, drop_last=False)
test_dl = make_loader(test_ds, shuffle=False, drop_last=False)

print(f"train : {len(train_ds):,} items over {len(train_dl):,} batches   "
      f"hours [{TRAIN_SLICE.start}, {TRAIN_SLICE.stop})")
print(f"val   : {len(val_ds):,} items over {len(val_dl):,} batches   "
      f"hours [{VAL_SLICE.start}, {VAL_SLICE.stop})")
print(f"test  : {len(test_ds):,} items over {len(test_dl):,} batches   "
      f"hours [{TEST_SLICE.start}, {TEST_SLICE.stop})")
print(f"gap between splits : {GAP}h (= longest horizon, prevents label leakage)")

for name, ds in (("train", train_ds), ("val", val_ds), ("test", test_ds)):
    assert len(ds) > 0, (
        f"The {name} split produced no windows. Raise panel_hours, or lower history_hours "
        f"/ the split fractions."
    )

## 7. Model — block-batched Graph WaveNet

Faithful to the architecture documented in
`docs/spatio-temporal-graph-risk-forecasting.md`. The graph operators act on one spatial
cluster at a time (§6.5), so the adjacency stays dense and small while the *network* the
model covers stays national.

```
input (B, T, N, F)
      │  1×1 conv  ───────────────────────────────────────┐
      ▼                                                    │ residual
┌──────────────────────────────────────────────────┐       │
│ block b, dilation 2^b                            │       │
│   temporal gated conv:  tanh(Wf*x) ⊙ σ(Wg*x)     │       │
│   diffusion graph conv:  Σ_k P^k x W_k           │       │
│                        + Σ_k Pᵀ^k x W'_k         │       │
│                        + A_adp x W_adp           │       │
│   dropout → residual → skip                      │───────┘
└──────────────────────────────────────────────────┘
      ▼ skip sum
   ReLU → 1×1 → ReLU → 1×1 → logits (B, N, |horizons|)
```

Three deliberate choices:

- **Bidirectional diffusion.** Risk propagates downstream with traffic, but congestion
  shockwaves propagate *upstream*. Using only the forward transition matrix systematically
  misses queue-driven risk.
- **Adaptive adjacency.** `A_adp = softmax(relu(E₁E₂ᵀ))` from learned node embeddings
  discovers couplings the topology does not encode — parallel routes that substitute for
  each other, corridors sharing a weather regime. This is Graph WaveNet's key idea, and
  the low-rank form keeps it tractable at national `N`.
- **Logits, not probabilities, from the network.** Calibration is applied explicitly after
  training (§10.2). A model that emits probabilities directly hides its own miscalibration.

The adaptive adjacency embeddings are held for **all** `N` nodes and indexed per block, so
a node keeps one identity across every batch it appears in — the embedding table is the
only part of the model that is national in size, and at `N × 16` floats it is trivial.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


class BlockDiffusionConv(nn.Module):
    """K-step bidirectional diffusion + learned adaptive adjacency, on one node block.

    `adj_*` are dense (nb, nb) transition matrices for the block; `node_idx` selects this
    block's rows from the national embedding table so node identity is shared across
    batches.
    """

    def __init__(self, in_ch: int, out_ch: int, k: int, n_nodes: int, emb_dim: int, dropout: float):
        super().__init__()
        self.k = k
        # identity + forward K steps + backward K steps + adaptive
        n_supports = 2 * k + 2
        self.mlp = nn.Conv2d(in_ch * n_supports, out_ch, kernel_size=(1, 1))
        self.dropout = dropout
        self.node_emb_src = nn.Parameter(torch.randn(n_nodes, emb_dim) * 0.05)
        self.node_emb_dst = nn.Parameter(torch.randn(n_nodes, emb_dim) * 0.05)

    @staticmethod
    def _propagate(adj: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        # x: (B, C, nb, T); mix over the node axis with a dense (nb, nb) operator.
        return torch.einsum("ij,bcjt->bcit", adj, x)

    def forward(self, x, adj_fwd, adj_bwd, node_idx):
        supports = [x]
        h = x
        for _ in range(self.k):
            h = self._propagate(adj_fwd, h)
            supports.append(h)
        h = x
        for _ in range(self.k):
            h = self._propagate(adj_bwd, h)
            supports.append(h)

        # Adaptive adjacency in fp32: the softmax is numerically fragile in bf16/fp16.
        with torch.autocast(device_type=x.device.type, enabled=False):
            e_src = self.node_emb_src[node_idx].float()
            e_dst = self.node_emb_dst[node_idx].float()
            adp = F.softmax(F.relu(e_src @ e_dst.T), dim=1)
        supports.append(self._propagate(adp.to(x.dtype), x))

        out = self.mlp(torch.cat(supports, dim=1))
        return F.dropout(out, self.dropout, training=self.training)


class GatedTemporalConv(nn.Module):
    """Dilated causal convolution with a tanh/sigmoid gate (WaveNet's temporal unit)."""

    def __init__(self, ch: int, kernel_size: int, dilation: int):
        super().__init__()
        self.filter = nn.Conv2d(ch, ch, kernel_size=(1, kernel_size), dilation=(1, dilation))
        self.gate = nn.Conv2d(ch, ch, kernel_size=(1, kernel_size), dilation=(1, dilation))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.tanh(self.filter(x)) * torch.sigmoid(self.gate(x))


class SpatioTemporalGraphWaveNet(nn.Module):
    def __init__(self, cfg: TrainConfig, n_nodes: int):
        super().__init__()
        self.cfg = cfg
        self.start = nn.Conv2d(cfg.in_features, cfg.hidden_channels, kernel_size=(1, 1))
        self.temporal = nn.ModuleList()
        self.spatial = nn.ModuleList()
        self.skips = nn.ModuleList()
        self.norms = nn.ModuleList()

        for b in range(cfg.n_blocks):
            self.temporal.append(GatedTemporalConv(cfg.hidden_channels, cfg.kernel_size, 2 ** b))
            self.spatial.append(
                BlockDiffusionConv(cfg.hidden_channels, cfg.hidden_channels,
                                   cfg.diffusion_steps, n_nodes,
                                   cfg.adaptive_embedding_dim, cfg.dropout)
            )
            self.skips.append(nn.Conv2d(cfg.hidden_channels, cfg.skip_channels, kernel_size=(1, 1)))
            self.norms.append(nn.BatchNorm2d(cfg.hidden_channels))

        self.head = nn.Sequential(
            nn.ReLU(),
            nn.Conv2d(cfg.skip_channels, cfg.end_channels, kernel_size=(1, 1)),
            nn.ReLU(),
            nn.Conv2d(cfg.end_channels, len(cfg.horizons), kernel_size=(1, 1)),
        )

    def forward(self, x, adj_fwd, adj_bwd, node_idx):
        # x arrives as (B, T, nb, F); the convolutions want (B, F, nb, T).
        x = x.permute(0, 3, 2, 1)
        h = self.start(x)
        skip_total = None

        for temporal, spatial, skip, norm in zip(self.temporal, self.spatial, self.skips, self.norms):
            residual = h
            h = temporal(h)
            h = spatial(h, adj_fwd, adj_bwd, node_idx)
            s = skip(h)
            skip_total = s if skip_total is None else skip_total[..., -s.size(-1):] + s
            h = h + residual[..., -h.size(-1):]
            h = norm(h)

        out = self.head(skip_total[..., -1:])   # (B, n_horizons, nb, 1)
        return out.squeeze(-1).permute(0, 2, 1)  # (B, nb, n_horizons)


# Node-index tensors for each block, cached on the GPU alongside its adjacency.
BLOCK_IDX = [torch.from_numpy(block).long().to(DEVICE) for block in NODE_BLOCKS]

model = SpatioTemporalGraphWaveNet(CFG, N_SEGMENTS).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
receptive_field = 1 + sum((CFG.kernel_size - 1) * 2 ** b for b in range(CFG.n_blocks))

print(f"parameters       : {n_params:,}")
print(f"receptive field  : {receptive_field}h of the {CFG.history_hours}h history window")
assert receptive_field <= CFG.history_hours, (
    "Receptive field exceeds the history window: raise history_hours or lower n_blocks."
)

## 8. Objective and metrics

Crash occurrence on a segment-hour is extremely rare. Three consequences drive everything
in this section:

- **Accuracy is meaningless.** Predicting "no crash" everywhere scores 99.99%.
- **ROC-AUC flatters.** With a 1e-4 positive rate, the false-positive axis is dominated by
  an enormous negative class; **PR-AUC** is the honest summary.
- **Operations care about ranking, not classification.** A control room can dispatch to
  *k* corridors per shift, so `recall@k` — the share of real crashes falling inside the top
  *k* ranked segments — is the metric that maps to the actual decision.

Focal loss down-weights the easy negatives that would otherwise swamp the gradient.
`alpha` tilts toward positives; `gamma` sharpens the focus on hard examples. Calibration
(Brier score, expected calibration error) is tracked separately, because focal loss is
known to *distort* probability calibration even while it improves ranking — hence the
explicit post-hoc temperature scaling in §9.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score


def focal_bce_with_logits(logits, targets, alpha: float, gamma: float, horizon_weights):
    """Per-horizon weighted focal loss on logits."""
    bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    p = torch.sigmoid(logits)
    p_t = p * targets + (1.0 - p) * (1.0 - targets)
    alpha_t = alpha * targets + (1.0 - alpha) * (1.0 - targets)
    loss = alpha_t * torch.pow(1.0 - p_t, gamma) * bce
    weights = torch.tensor(horizon_weights, device=logits.device, dtype=loss.dtype)
    return (loss.mean(dim=(0, 1)) * weights).sum() / weights.sum()


def expected_calibration_error(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15) -> float:
    """Binned |confidence - accuracy|, weighted by bin population."""
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(probs, edges) - 1, 0, n_bins - 1)
    ece = 0.0
    for b in range(n_bins):
        mask = idx == b
        if not mask.any():
            continue
        ece += mask.mean() * abs(probs[mask].mean() - labels[mask].mean())
    return float(ece)


def recall_at_k(probs: np.ndarray, labels: np.ndarray, k: int) -> float:
    """Share of true positives captured by the top-k ranked segment-hours."""
    if labels.sum() == 0:
        return float("nan")
    top = np.argpartition(-probs, min(k, probs.size - 1))[:k]
    return float(labels[top].sum() / labels.sum())


def evaluate_arrays(probs: np.ndarray, labels: np.ndarray, top_k: int = 500) -> dict:
    out = {
        "positives": int(labels.sum()),
        "positive_rate": float(labels.mean()),
        "brier": float(np.mean((probs - labels) ** 2)),
        "ece": expected_calibration_error(probs, labels),
        f"recall@{top_k}": recall_at_k(probs, labels, top_k),
    }
    if 0 < labels.sum() < labels.size:
        out["pr_auc"] = float(average_precision_score(labels, probs))
        out["roc_auc"] = float(roc_auc_score(labels, probs))
        out["lift_over_base"] = out["pr_auc"] / max(out["positive_rate"], 1e-12)
    return out

## 9. Training loop

Built for a runtime that can vanish. Colab reclaims sessions; a 40-minute run that loses
everything on disconnect is a 40-minute run you did twice. Every epoch checkpoints
model + optimiser + scheduler + scaler + epoch index, and the loop resumes from the last
checkpoint automatically.

Other choices worth knowing:

- **AMP** in bf16 where supported (no loss scaling needed, no overflow drama); fp16 with a
  `GradScaler` on a T4.
- **Gradient clipping** before the optimiser step — gated activations plus rare positives
  produce occasional very large gradients.
- **Cosine schedule with warmup**; the adaptive-adjacency embeddings are unstable if the
  first few hundred steps run at full learning rate.
- **Early stopping on validation PR-AUC**, not on loss. Loss keeps improving after ranking
  quality has plateaued.

In [ ]:
def make_scheduler(optimizer, total_steps: int, warmup_frac: float):
    warmup = max(1, int(total_steps * warmup_frac))

    def lr_lambda(step: int) -> float:
        if step < warmup:
            return step / warmup
        progress = (step - warmup) / max(1, total_steps - warmup)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def save_checkpoint(path: Path, model, optimizer, scheduler, scaler, epoch: int, best: float, history: list):
    torch.save(
        {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict() if scaler is not None else None,
            "epoch": epoch,
            "best_metric": best,
            "history": history,
            "config": asdict(CFG),
            "config_hash": CFG.hash(),
            "git_sha": GIT_SHA,
            "seed": SEED,
        },
        path,
    )


@torch.no_grad()
def collect_predictions(loader, model, max_batches: int | None = None):
    model.eval()
    probs, labels = [], []
    for i, (xb, yb, block_id) in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        xb = xb.to(DEVICE, non_blocking=True)
        adj_fwd, adj_bwd = BLOCK_ADJ[block_id]
        with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(xb, adj_fwd, adj_bwd, BLOCK_IDX[block_id])
        probs.append(torch.sigmoid(logits.float()).cpu().numpy().reshape(-1, len(CFG.horizons)))
        labels.append(yb.numpy().reshape(-1, len(CFG.horizons)))
    return np.concatenate(probs), np.concatenate(labels)


AMP_DTYPE = torch.bfloat16 if CFG.amp_dtype == "bfloat16" else torch.float16
USE_AMP = DEVICE.type == "cuda"
if USE_AMP and AMP_DTYPE is torch.bfloat16 and not torch.cuda.is_bf16_supported():
    print("bf16 unsupported on this GPU — falling back to fp16.")
    AMP_DTYPE = torch.float16

print(f"AMP enabled : {USE_AMP} ({AMP_DTYPE})")

In [ ]:
CKPT_PATH = CKPT_DIR / f"stgnn_{CFG.hash()}.pt"
BEST_PATH = CKPT_DIR / f"stgnn_{CFG.hash()}_best.pt"

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
steps_per_epoch = max(1, len(train_dl) // CFG.grad_accumulation)
scheduler = make_scheduler(optimizer, steps_per_epoch * CFG.epochs, CFG.warmup_frac)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE is torch.float16)

start_epoch, best_metric, history = 0, -math.inf, []
if CKPT_PATH.exists():
    state = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    if state.get("config_hash") == CFG.hash():
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        if state.get("scaler") is not None:
            scaler.load_state_dict(state["scaler"])
        start_epoch = state["epoch"] + 1
        best_metric = state["best_metric"]
        history = state["history"]
        print(f"Resumed from epoch {start_epoch} (best PR-AUC {best_metric:.5f})")
    else:
        print("Checkpoint found but the config changed — starting fresh.")

if CFG.use_torch_compile and hasattr(torch, "compile"):
    model = torch.compile(model)
    print("torch.compile enabled (expect a slow first step)")

In [ ]:
epochs_without_improvement = 0

for epoch in range(start_epoch, CFG.epochs):
    model.train()
    epoch_loss, n_batches = 0.0, 0
    t0 = time.time()
    optimizer.zero_grad(set_to_none=True)

    for step, (xb, yb, block_id) in enumerate(train_dl):
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        adj_fwd, adj_bwd = BLOCK_ADJ[block_id]

        with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = model(xb, adj_fwd, adj_bwd, BLOCK_IDX[block_id])
            loss = focal_bce_with_logits(
                logits.float(), yb, CFG.focal_alpha, CFG.focal_gamma, CFG.horizon_weights
            ) / CFG.grad_accumulation

        scaler.scale(loss).backward()

        if (step + 1) % CFG.grad_accumulation == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip_norm)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        epoch_loss += float(loss.item()) * CFG.grad_accumulation
        n_batches += 1

    val_probs, val_labels = collect_predictions(val_dl, model)
    metrics = evaluate_arrays(val_probs.reshape(-1), val_labels.reshape(-1))
    val_pr_auc = metrics.get("pr_auc", float("nan"))

    record = {
        "epoch": epoch,
        "train_loss": epoch_loss / max(1, n_batches),
        "seconds": round(time.time() - t0, 1),
        "lr": scheduler.get_last_lr()[0],
        **metrics,
    }
    history.append(record)
    print(
        f"epoch {epoch:>3} | loss {record['train_loss']:.5f} | "
        f"val PR-AUC {val_pr_auc:.5f} | Brier {metrics['brier']:.5f} | "
        f"ECE {metrics['ece']:.4f} | {record['seconds']:.0f}s"
    )

    save_checkpoint(CKPT_PATH, model, optimizer, scheduler, scaler, epoch, best_metric, history)

    if val_pr_auc > best_metric:
        best_metric = val_pr_auc
        epochs_without_improvement = 0
        save_checkpoint(BEST_PATH, model, optimizer, scheduler, scaler, epoch, best_metric, history)
        print(f"          new best -> {BEST_PATH.name}")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= CFG.early_stop_patience:
            print(f"Early stop: no improvement for {CFG.early_stop_patience} epochs.")
            break

print(f"\nbest validation PR-AUC : {best_metric:.5f}")

In [ ]:
# Training curves. Loss and PR-AUC diverging is the signal that focal loss is
# trading calibration for ranking — the temperature scaling below is the remedy.
import matplotlib.pyplot as plt

if history:
    epochs_axis = [h["epoch"] for h in history]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(epochs_axis, [h["train_loss"] for h in history])
    axes[0].set_title("training focal loss")
    axes[1].plot(epochs_axis, [h.get("pr_auc", np.nan) for h in history], color="#0a7")
    axes[1].set_title("validation PR-AUC")
    axes[2].plot(epochs_axis, [h["ece"] for h in history], color="#a30")
    axes[2].set_title("validation ECE (lower is better)")
    for ax in axes:
        ax.set_xlabel("epoch")
        ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

## 10. Evaluation, calibration and ablation

### 10.1 Held-out test performance

The test slice has never been touched — not by the scaler, not by early stopping, not by
threshold selection. It gets read **once**. If you find yourself tuning after seeing these
numbers, you have converted your test set into a second validation set and must re-split.

### 10.2 Calibration

Operational decisions are taken against a probability threshold, so a model whose "0.6"
means 0.2 in practice will misallocate patrols no matter how good its ranking is.
Temperature scaling fits a single scalar on the **validation** set — one parameter, so it
cannot overfit meaningfully — and typically undoes most of the distortion focal loss
introduces.

### 10.3 Ablations

A headline score means nothing without knowing which component earns it. Four ablations
answer the questions a reviewer will actually ask: does the graph matter, does the
adaptive adjacency matter, does weather matter, does telematics matter?

In [ ]:
# Load the best checkpoint and read the test slice once.
best_state = torch.load(BEST_PATH, map_location=DEVICE, weights_only=False)
eval_model = SpatioTemporalGraphWaveNet(CFG, N_SEGMENTS).to(DEVICE)
eval_model.load_state_dict(best_state["model"])
eval_model.eval()

test_probs, test_labels = collect_predictions(test_dl, eval_model)
print(f"test predictions : {test_probs.shape}")

print("\nper-horizon test metrics")
print("-" * 72)
per_horizon = {}
for hi, horizon in enumerate(CFG.horizons):
    m = evaluate_arrays(test_probs[..., hi].reshape(-1), test_labels[..., hi].reshape(-1))
    per_horizon[f"{horizon}h"] = m
    print(f"{horizon:>3}h | PR-AUC {m.get('pr_auc', float('nan')):.5f} | "
          f"ROC-AUC {m.get('roc_auc', float('nan')):.4f} | Brier {m['brier']:.5f} | "
          f"ECE {m['ece']:.4f} | recall@500 {m['recall@500']:.3f} | "
          f"lift {m.get('lift_over_base', float('nan')):.1f}x")

In [ ]:
# Temperature scaling, fitted on validation only.
def fit_temperature(val_logits: torch.Tensor, val_targets: torch.Tensor) -> float:
    log_t = torch.zeros(1, requires_grad=True, device=val_logits.device)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=60)

    def closure():
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(val_logits / log_t.exp(), val_targets)
        loss.backward()
        return loss

    opt.step(closure)
    return float(log_t.exp().item())


@torch.no_grad()
def collect_logits(loader, model):
    model.eval()
    logits, labels = [], []
    for xb, yb, block_id in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        adj_fwd, adj_bwd = BLOCK_ADJ[block_id]
        with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            out = model(xb, adj_fwd, adj_bwd, BLOCK_IDX[block_id])
        logits.append(out.float().cpu().reshape(-1, len(CFG.horizons)))
        labels.append(yb.reshape(-1, len(CFG.horizons)))
    return torch.cat(logits), torch.cat(labels)


val_logits, val_targets = collect_logits(val_dl, eval_model)
TEMPERATURE = fit_temperature(val_logits.reshape(-1, 1).to(DEVICE), val_targets.reshape(-1, 1).to(DEVICE))
print(f"fitted temperature : {TEMPERATURE:.4f}  (>1 means the model was over-confident)")

test_logits, _ = collect_logits(test_dl, eval_model)
calibrated_probs = torch.sigmoid(test_logits / TEMPERATURE).numpy()

flat_labels = test_labels.reshape(-1)
before = evaluate_arrays(test_probs.reshape(-1), flat_labels)
after = evaluate_arrays(calibrated_probs.reshape(-1), flat_labels)
print(f"ECE   {before['ece']:.4f} -> {after['ece']:.4f}")
print(f"Brier {before['brier']:.5f} -> {after['brier']:.5f}")
print(f"PR-AUC unchanged by a monotone transform: "
      f"{before.get('pr_auc', float('nan')):.5f} -> {after.get('pr_auc', float('nan')):.5f}")

In [ ]:
# Reliability diagram: perfectly calibrated predictions sit on the diagonal.
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(5.5, 5.5))
for label, probs in (("raw", test_probs.reshape(-1)), ("temperature-scaled", calibrated_probs.reshape(-1))):
    if 0 < flat_labels.sum() < flat_labels.size:
        frac_pos, mean_pred = calibration_curve(flat_labels, probs, n_bins=12, strategy="quantile")
        ax.plot(mean_pred, frac_pos, marker="o", label=label)
ax.plot([0, 1], [0, 1], "k--", linewidth=1, label="perfect")
ax.set_xlabel("predicted probability")
ax.set_ylabel("observed frequency")
ax.set_title("Reliability diagram (test slice)")
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Bootstrap confidence intervals. A single point estimate on a rare-event problem is
# not a result; the spread across resamples is what tells you whether a delta is real.
def bootstrap_metric(probs: np.ndarray, labels: np.ndarray, n_boot: int = 200, seed: int = SEED):
    boot_rng = np.random.default_rng(seed)
    scores = []
    size = probs.size
    for _ in range(n_boot):
        pick = boot_rng.integers(0, size, size=size)
        lab = labels[pick]
        if 0 < lab.sum() < lab.size:
            scores.append(average_precision_score(lab, probs[pick]))
    if not scores:
        return float("nan"), float("nan"), float("nan")
    arr = np.array(scores)
    return float(arr.mean()), float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5))


# Subsample for tractability — full-panel bootstrap on a national network is hours of CPU.
sub = np.random.default_rng(SEED).choice(flat_labels.size, size=min(400_000, flat_labels.size), replace=False)
mean_ap, lo, hi = bootstrap_metric(calibrated_probs.reshape(-1)[sub], flat_labels[sub])
print(f"test PR-AUC : {mean_ap:.5f}  95% CI [{lo:.5f}, {hi:.5f}]")

In [ ]:
# --- ablations -----------------------------------------------------------
# Each variant is trained briefly from scratch; the comparison is relative, so a short
# budget is legitimate as long as every arm gets the same one.
ABLATION_EPOCHS = 3


def train_variant(name: str, *, disable_graph=False, disable_adaptive=False,
                  zero_features: tuple[int, ...] = ()) -> dict:
    seed_everything(SEED, STRICT_DETERMINISM)
    variant = SpatioTemporalGraphWaveNet(CFG, N_SEGMENTS).to(DEVICE)
    if disable_adaptive:
        for module in variant.modules():
            if isinstance(module, BlockDiffusionConv):
                module.node_emb_src.requires_grad_(False)
                module.node_emb_dst.requires_grad_(False)
                module.node_emb_src.data.zero_()
                module.node_emb_dst.data.zero_()

    def adjacency_for(block_id: int):
        if not disable_graph:
            return BLOCK_ADJ[block_id]
        nb = len(NODE_BLOCKS[block_id])
        eye = torch.eye(nb, device=DEVICE)
        return eye, eye

    opt = torch.optim.AdamW(variant.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    sch = make_scheduler(opt, max(1, len(train_dl)) * ABLATION_EPOCHS, CFG.warmup_frac)
    sc = torch.amp.GradScaler("cuda", enabled=USE_AMP and AMP_DTYPE is torch.float16)

    for _ in range(ABLATION_EPOCHS):
        variant.train()
        for xb, yb, block_id in train_dl:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            if zero_features:
                xb = xb.clone()
                xb[..., list(zero_features)] = 0.0
            a_fwd, a_bwd = adjacency_for(block_id)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                loss = focal_bce_with_logits(
                    variant(xb, a_fwd, a_bwd, BLOCK_IDX[block_id]).float(), yb,
                    CFG.focal_alpha, CFG.focal_gamma, CFG.horizon_weights,
                )
            sc.scale(loss).backward()
            sc.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(variant.parameters(), CFG.grad_clip_norm)
            sc.step(opt)
            sc.update()
            sch.step()

    variant.eval()
    probs, labels = [], []
    with torch.no_grad():
        for xb, yb, block_id in val_dl:
            xb = xb.to(DEVICE, non_blocking=True)
            if zero_features:
                xb = xb.clone()
                xb[..., list(zero_features)] = 0.0
            a_fwd, a_bwd = adjacency_for(block_id)
            with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                out = variant(xb, a_fwd, a_bwd, BLOCK_IDX[block_id])
            probs.append(torch.sigmoid(out.float()).cpu().numpy().reshape(-1, len(CFG.horizons)))
            labels.append(yb.numpy().reshape(-1, len(CFG.horizons)))
    del variant
    torch.cuda.empty_cache()
    return {"variant": name, **evaluate_arrays(np.concatenate(probs).reshape(-1),
                                               np.concatenate(labels).reshape(-1))}


RUN_ABLATIONS = False  # flip on when you need the table; it costs ~4x one short training run

if RUN_ABLATIONS:
    rows = [
        train_variant("full model"),
        train_variant("no graph (identity adjacency)", disable_graph=True),
        train_variant("no adaptive adjacency", disable_adaptive=True),
        train_variant("no weather", zero_features=(3, 4, 5)),
        train_variant("no telematics", zero_features=(1,)),
    ]
    import pandas as pd
    from IPython.display import display

    ablation_df = pd.DataFrame(rows)[["variant", "pr_auc", "brier", "ece", "recall@500"]]
    display(ablation_df.round(5))
    print()
    print("Read this table as: how much does each component earn? A variant whose PR-AUC")
    print("matches the full model is a component you cannot justify keeping.")
else:
    print("Ablations skipped. Set RUN_ABLATIONS = True before writing up results —")
    print("a reported score without an ablation table is not a finding.")

## 11. National-scale spatial kernel density estimation

KDE turns discrete crash points into a continuous risk surface. The naive formulation is

$$\hat{f}(\mathbf{s}) = \frac{1}{n h^2}\sum_{i=1}^{n} K\!\left(\frac{\|\mathbf{s}-\mathbf{s}_i\|}{h}\right)$$

evaluated at every raster cell — which for a national grid is `G × M` distance
computations. At 100 m cells over a country and 10⁵ crash points that is on the order of
10¹¹ pairs: hours of GPU time for a surface that is ~99.9% zero.

**The trick is to invert the loop.** A Gaussian kernel truncated at 3σ has negligible mass
beyond that radius, so instead of asking every cell which points are near it, each point
*scatters* its small footprint into the raster. Cost drops to `M × w²` where `w` is the
truncation window in cells — linear in the number of crashes and entirely independent of
country size. On a GPU this is a single `index_add_`.

### Bandwidth is the analysis

The bandwidth `h` is not a rendering parameter, it is the spatial scale of the hypothesis.
Too small and every cluster is a single crash; too large and a city smears into its region.
Silverman's rule is the reflex default, but it assumes an approximately normal
distribution — a road network is emphatically not that, so for operational screening,
choose `h` from the intervention's own footprint: the length of corridor a countermeasure
actually treats, typically 250–500 m.

### Planar versus network-constrained

Planar KDE leaks density across a river or a motorway embankment that no vehicle can cross.
**Network-constrained KDE** evaluates density only at points on the network and measures
distance *along* it. Both are computed below: the raster for rendering and communication,
the per-segment density for anything feeding a decision.

In [ ]:
# Crash point cloud, weighted by the crashes observed in the panel.
crash_hours, crash_segments = np.nonzero(CRASH_COUNTS > 0)
crash_weights = CRASH_COUNTS[crash_hours, crash_segments].astype(np.float32)
crash_xy = XY_M[crash_segments]

# Jitter within the segment so co-located events do not stack on one raster cell.
jitter_rng = np.random.default_rng(SEED + 1)
crash_xy = crash_xy + jitter_rng.normal(0.0, 40.0, size=crash_xy.shape)

print(f"crash events : {len(crash_xy):,} (total weight {crash_weights.sum():,.0f})")


def silverman_bandwidth(points: np.ndarray) -> float:
    """Silverman's rule of thumb, 2-D, for reference against the chosen bandwidth."""
    n = len(points)
    sigma = float(np.mean([points[:, 0].std(), points[:, 1].std()]))
    return float(sigma * (4.0 / (4.0 * n)) ** (1.0 / 6.0))


print(f"Silverman h  : {silverman_bandwidth(crash_xy):,.0f} m")
print(f"configured h : {CFG.kde_bandwidth_m:,.0f} m  <- chosen from intervention footprint")

In [ ]:
def gpu_scatter_kde(points_xy: np.ndarray, weights: np.ndarray, cfg: TrainConfig,
                    device: torch.device, chunk: int = 50_000):
    """Truncated-Gaussian KDE by scattering each point's footprint into the raster.

    Linear in the number of points and independent of the extent of the study area,
    which is what makes national coverage tractable.
    """
    cell = cfg.kde_cell_m
    h = cfg.kde_bandwidth_m
    radius_cells = int(math.ceil(cfg.kde_truncation_sigmas * h / cell))
    pad = radius_cells * cell

    x_min, y_min = points_xy.min(axis=0) - pad
    x_max, y_max = points_xy.max(axis=0) + pad
    n_x = int(math.ceil((x_max - x_min) / cell))
    n_y = int(math.ceil((y_max - y_min) / cell))

    raster = torch.zeros(n_y * n_x, device=device, dtype=torch.float32)

    # Precompute the kernel footprint once — it is identical for every point.
    offsets = torch.arange(-radius_cells, radius_cells + 1, device=device, dtype=torch.float32)
    dy, dx = torch.meshgrid(offsets, offsets, indexing="ij")
    kernel = torch.exp(-((dx * cell) ** 2 + (dy * cell) ** 2) / (2.0 * h ** 2))
    kernel = kernel / (2.0 * math.pi * h ** 2)          # density units: events per m²
    kernel_flat = kernel.reshape(-1)
    off_y = dy.reshape(-1).long()
    off_x = dx.reshape(-1).long()

    for start in range(0, len(points_xy), chunk):
        stop = min(start + chunk, len(points_xy))
        px = torch.from_numpy(points_xy[start:stop, 0]).to(device)
        py = torch.from_numpy(points_xy[start:stop, 1]).to(device)
        pw = torch.from_numpy(weights[start:stop]).to(device)

        col = ((px - x_min) / cell).long()
        row = ((py - y_min) / cell).long()

        rows = row[:, None] + off_y[None, :]
        cols = col[:, None] + off_x[None, :]
        valid = (rows >= 0) & (rows < n_y) & (cols >= 0) & (cols < n_x)

        flat_idx = (rows * n_x + cols)[valid]
        contrib = (pw[:, None] * kernel_flat[None, :])[valid]
        raster.index_add_(0, flat_idx, contrib)

    return raster.reshape(n_y, n_x).cpu().numpy(), (x_min, y_min, cell, n_x, n_y)


t0 = time.time()
KDE_RASTER, KDE_GEO = gpu_scatter_kde(crash_xy, crash_weights, CFG, DEVICE)
x_min, y_min, cell, n_x, n_y = KDE_GEO

print(f"raster      : {n_y:,} x {n_x:,} cells at {cell:.0f} m "
      f"({n_y * n_x * 4 / 1024 ** 2:,.0f} MiB)")
print(f"extent      : {n_x * cell / 1000:,.0f} km x {n_y * cell / 1000:,.0f} km")
print(f"elapsed     : {time.time() - t0:.1f}s on {DEVICE}")
print(f"peak density: {KDE_RASTER.max() * 1e6:.3f} events per km^2")
print(f"non-zero    : {(KDE_RASTER > 0).mean() * 100:.1f}% of cells")

In [ ]:
# Network-constrained density: evaluated on the network, with distance measured
# through the graph rather than across open country.
def network_constrained_density(seg_xy: np.ndarray, points_xy: np.ndarray,
                                weights: np.ndarray, cfg: TrainConfig,
                                edge_index: torch.Tensor, edge_weight: torch.Tensor,
                                hops: int = 2) -> np.ndarray:
    radius = cfg.kde_truncation_sigmas * cfg.kde_bandwidth_m
    point_tree = cKDTree(points_xy)

    density = np.zeros(len(seg_xy), dtype=np.float64)
    neighbours = point_tree.query_ball_point(seg_xy, r=radius, workers=-1)
    norm = 1.0 / (2.0 * math.pi * cfg.kde_bandwidth_m ** 2)
    for i, idx in enumerate(neighbours):
        if not idx:
            continue
        d = np.linalg.norm(points_xy[idx] - seg_xy[i], axis=1)
        density[i] = norm * float(np.sum(weights[idx] * np.exp(-(d ** 2) / (2.0 * cfg.kde_bandwidth_m ** 2))))

    # Smooth along the network so density follows connectivity, not open space. The
    # already-normalised Gaussian edge weights mean a nearby neighbour pulls harder than
    # a distant one — this is the step that makes the surface network-constrained rather
    # than merely network-sampled.
    src_np = edge_index[0].numpy()
    dst_np = edge_index[1].numpy()
    w_np = edge_weight.numpy().astype(np.float64)
    for _ in range(hops):
        neighbour_sum = np.zeros_like(density)
        weight_sum = np.zeros_like(density)
        np.add.at(neighbour_sum, src_np, w_np * density[dst_np])
        np.add.at(weight_sum, src_np, w_np)
        weight_sum[weight_sum == 0] = 1.0
        density = 0.65 * density + 0.35 * (neighbour_sum / weight_sum)

    return density


t0 = time.time()
SEGMENT_DENSITY = network_constrained_density(
    XY_M, crash_xy, crash_weights, CFG, EDGE_INDEX, EDGE_WEIGHT
)
print(f"network-constrained density over {N_SEGMENTS:,} segments in {time.time() - t0:.1f}s")
print(f"  median {np.median(SEGMENT_DENSITY) * 1e6:.4f} / km^2, "
      f"p99 {np.percentile(SEGMENT_DENSITY, 99) * 1e6:.4f} / km^2")

# Planar-vs-network disagreement is itself a finding: where they differ most, the planar
# surface is leaking density across barriers the network does not cross.
row_idx = np.clip(((XY_M[:, 1] - y_min) / cell).astype(int), 0, n_y - 1)
col_idx = np.clip(((XY_M[:, 0] - x_min) / cell).astype(int), 0, n_x - 1)
PLANAR_AT_SEGMENT = KDE_RASTER[row_idx, col_idx]

mask = PLANAR_AT_SEGMENT > 0
if mask.sum() > 10:
    corr = float(np.corrcoef(PLANAR_AT_SEGMENT[mask], SEGMENT_DENSITY[mask])[0, 1])
    print(f"  planar vs network-constrained correlation : {corr:.3f}")
    print("  (a low correlation means barriers matter here — trust the network surface)")

In [ ]:
# Save the KDE products in formats the rest of the platform can read.
# A national raster at 100 m cells is ~130M cells (500+ MiB raw) and is ~99% zero,
# because density collapses to the network. Compressed storage turns that into a few
# MiB; uncompressed it is the single largest artifact of the run for no benefit.
raster_path = KDE_DIR / f"{CFG.network_label}_kde_raster.npz"
np.savez_compressed(raster_path, density=KDE_RASTER.astype(np.float32))
raw_mib = KDE_RASTER.nbytes / 1024 ** 2
print(f"raster: {raw_mib:,.0f} MiB raw -> {raster_path.stat().st_size / 1024 ** 2:,.1f} MiB compressed")
print('load it back with: np.load(path)["density"]')
(KDE_DIR / f"{CFG.network_label}_kde_geotransform.json").write_text(
    json.dumps(
        {
            "x_min_m": float(x_min), "y_min_m": float(y_min),
            "cell_size_m": float(cell), "n_cols": int(n_x), "n_rows": int(n_y),
            "crs_proj4": METRIC_CRS.to_proj4(),
            "bandwidth_m": CFG.kde_bandwidth_m,
            "truncation_sigmas": CFG.kde_truncation_sigmas,
            "kernel": "gaussian_truncated",
            "units": "events_per_square_metre",
            "synthetic": bool(IS_SYNTHETIC),
        },
        indent=2,
    ),
    encoding="utf-8",
)

import pandas as pd

density_df = pd.DataFrame(
    {
        "segment_id": NET["segment_id"],
        "latitude": NET["latitude"],
        "longitude": NET["longitude"],
        "functional_class": NET["functional_class"],
        "kde_density_per_km2": SEGMENT_DENSITY * 1e6,
        "planar_density_per_km2": PLANAR_AT_SEGMENT * 1e6,
    }
)
density_df.to_csv(KDE_DIR / f"{CFG.network_label}_segment_density.csv", index=False)
print(f"wrote {KDE_DIR}")
for path in sorted(KDE_DIR.iterdir()):
    print(f"  {path.name}  ({path.stat().st_size / 1024:,.0f} KiB)")

## 12. Export through the Layer 2 contract

The trained model's output is converted into the repository's own dataclasses and written
by `export_st_gnn_artifacts` — the same function the CLI calls. Nothing here invents a file
format, so the reports hub, the API and the operations board all read this output without
any special-casing for "the notebook run".

Alert tiers use the thresholds the engine documents: `critical ≥ 0.75`, `high ≥ 0.55`,
`elevated ≥ 0.35`, otherwise `normal`.

In [ ]:
def tier_for(probability: float) -> str:
    if probability >= 0.75:
        return "critical"
    if probability >= 0.55:
        return "high"
    if probability >= 0.35:
        return "elevated"
    return "normal"


def contributing_factors(seg_idx: int, hour_idx: int) -> list[str]:
    """Human-readable drivers, read back off the un-normalised panel semantics."""
    factors = []
    raw = X_PANEL[hour_idx, seg_idx] * sd_t + mu_t
    if raw[0] > 0.5:
        factors.append("recent_crash_history")
    if raw[1] > 0.6:
        factors.append("elevated_near_miss_rate")
    if raw[2] < 0.65:
        factors.append("congestion_speed_drop")
    if raw[3] > 0.15 or raw[4] > 0.5:
        factors.append("adverse_weather_surface")
    if raw[8] > 0.5:
        factors.append("peak_hour_exposure")
    if SEGMENT_DENSITY[seg_idx] > np.percentile(SEGMENT_DENSITY, 95):
        factors.append("kde_hotspot_top5pct")
    return factors or ["baseline_exposure"]


# Forecast the whole national network from the most recent window, one block at a time.
last_start = int(test_ds.starts[-1])
history = X_PANEL[last_start - CFG.history_hours:last_start]

latest_probs = np.zeros((N_SEGMENTS, len(CFG.horizons)), dtype=np.float32)
eval_model.eval()
with torch.no_grad():
    for block_id, block in enumerate(NODE_BLOCKS):
        window = torch.from_numpy(np.ascontiguousarray(history[:, block])).unsqueeze(0).to(DEVICE)
        adj_fwd, adj_bwd = BLOCK_ADJ[block_id]
        with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
            logits = eval_model(window, adj_fwd, adj_bwd, BLOCK_IDX[block_id])
        latest_probs[block] = torch.sigmoid(logits.float() / TEMPERATURE).cpu().numpy()[0]

print(f"forecast produced for all {N_SEGMENTS:,} segments across "
      f"{len(NODE_BLOCKS)} spatial blocks")

forecast_ts = datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ")

# Cap exported rows: a national CSV of every segment x horizon is unwieldy, and the
# operational question is about the ranked head of the distribution.
EXPORT_TOP_N = 5_000
forecasts: list[SegmentRiskForecast] = []
alerts: list[HighRiskCorridorAlert] = []

for hi, horizon in enumerate(CFG.horizons):
    probs_h = latest_probs[:, hi]
    ranked = np.argsort(-probs_h)[:EXPORT_TOP_N]
    contagion = probs_h.copy()
    # Spatial contagion: how much of a segment's risk is shared by its neighbours.
    neighbour_mean = np.zeros(N_SEGMENTS)
    counts = np.zeros(N_SEGMENTS)
    np.add.at(neighbour_mean, src, probs_h[dst])
    np.add.at(counts, src, 1.0)
    counts[counts == 0] = 1.0
    neighbour_mean /= counts

    for rank, seg_idx in enumerate(ranked):
        probability = float(probs_h[seg_idx])
        tier = tier_for(probability)
        forecasts.append(
            SegmentRiskForecast(
                segment_id=str(NET["segment_id"][seg_idx]),
                horizon_hours=int(horizon),
                forecast_timestamp=forecast_ts,
                crash_probability=probability,
                expected_risk_intensity=float(min(100.0, probability * 100.0)),
                spatial_contagion_index=float(neighbour_mean[seg_idx]),
                alert_tier=tier,
                contributing_factors=contributing_factors(int(seg_idx), last_start - 1),
            )
        )
        if probability >= CFG.alert_threshold:
            drivers = contributing_factors(int(seg_idx), last_start - 1)
            alerts.append(
                HighRiskCorridorAlert(
                    alert_id=f"stgnn-{RUN_ID}-{horizon}h-{rank:05d}",
                    segment_id=str(NET["segment_id"][seg_idx]),
                    horizon_hours=int(horizon),
                    alert_tier=tier,
                    crash_probability=probability,
                    primary_threat=drivers[0],
                    recommended_action=(
                        "dispatch_patrol_and_vms" if tier == "critical"
                        else "increase_monitoring_and_signal_review"
                    ),
                )
            )

print(f"forecast rows : {len(forecasts):,}")
print(f"alerts        : {len(alerts):,} at threshold {CFG.alert_threshold}")

In [ ]:
export_payload = export_st_gnn_artifacts(
    forecasts=forecasts,
    alerts=alerts,
    output_dir=REPORT_DIR,
    network_label=CFG.network_label,
)

print("artifacts written by the Layer 2 exporter:")
for key, value in export_payload.items():
    print(f"  {key:<28} {value}")

In [ ]:
# Model card: everything needed to attribute, reproduce or retire this model.
model_card = {
    "run_id": RUN_ID,
    "created_at": datetime.now(UTC).isoformat(),
    "git_sha": GIT_SHA,
    "repo_ref": REPO_REF,
    "engine_version": ST_GNN_ENGINE_VERSION,
    "contract_version": ST_GNN_CONTRACT_VERSION,
    "config": asdict(CFG),
    "config_hash": CFG.hash(),
    "seed": SEED,
    "strict_determinism": STRICT_DETERMINISM,
    "hardware": (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"),
    "library_versions": {
        "torch": torch.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
    "data": {
        "synthetic": bool(IS_SYNTHETIC),
        "n_segments": int(N_SEGMENTS),
        "panel_hours": int(CFG.panel_hours),
        "train_hours": [int(TRAIN_SLICE.start), int(TRAIN_SLICE.stop)],
        "val_hours": [int(VAL_SLICE.start), int(VAL_SLICE.stop)],
        "test_hours": [int(TEST_SLICE.start), int(TEST_SLICE.stop)],
        "split_gap_hours": int(GAP),
    },
    "metrics": {"per_horizon_test": per_horizon, "validation_best_pr_auc": float(best_metric)},
    "calibration": {"method": "temperature_scaling", "temperature": TEMPERATURE,
                    "ece_before": before["ece"], "ece_after": after["ece"]},
    "kde": {"bandwidth_m": CFG.kde_bandwidth_m, "cell_m": CFG.kde_cell_m,
            "kernel": "gaussian_truncated", "crs": METRIC_CRS.to_proj4()},
    "limitations": [
        "Synthetic network in use — not a safety finding." if IS_SYNTHETIC
        else "Trained on the supplied national network extract.",
        "Rare-event regime: PR-AUC and recall@k are the decision-relevant metrics, not accuracy.",
        "Forecasts are correlational; causal countermeasure effects belong to causal_ml_engine.",
        "Planar KDE leaks across physical barriers — use the network-constrained density for decisions.",
    ],
}

card_path = CKPT_DIR / "model_card.json"
card_path.write_text(json.dumps(model_card, indent=2, default=str), encoding="utf-8")
print(f"model card -> {card_path}")
print(json.dumps({k: model_card[k] for k in ("run_id", "git_sha", "config_hash", "hardware")}, indent=2))

## 13. Interactive cloud rendering

Two renderers, for two different audiences.

**Folium / Leaflet** — raster tiles, works everywhere, embeds in a report, survives being
emailed. Right for a bulletin or an appendix figure. It will struggle beyond a few
thousand markers, so the map below draws the ranked head of the distribution, not the whole
network.

**Kepler.gl** — WebGL, comfortable with hundreds of thousands of rows, with time playback
and 3-D extrusion. Right for exploring a national surface interactively. Its notebook
widget is unreliable inside Colab's iframe sandbox, so we save to a self-contained HTML
file and open it — which is also what you would hand to a stakeholder.

Both draw the **network-constrained** density, not the planar raster, for anything that
implies a decision.

In [ ]:
import folium
from folium.plugins import HeatMap, MarkerCluster

TIER_COLOURS = {"critical": "#8B0000", "high": "#E02B20", "elevated": "#F0A32B", "normal": "#4B5563"}

centre = [float(np.median(NET["latitude"])), float(np.median(NET["longitude"]))]
fmap = folium.Map(location=centre, zoom_start=7, tiles="cartodbpositron", control_scale=True)

# Layer 1 — KDE heat surface, sampled at segment locations so it follows the network.
top_density = np.argsort(-SEGMENT_DENSITY)[:20_000]
heat_rows = [
    [float(NET["latitude"][i]), float(NET["longitude"][i]), float(SEGMENT_DENSITY[i])]
    for i in top_density
]
heat_layer = folium.FeatureGroup(name="Crash KDE (network-constrained)", show=True)
HeatMap(heat_rows, radius=12, blur=18, min_opacity=0.25).add_to(heat_layer)
heat_layer.add_to(fmap)

# Layer 2 — forecast alerts for the shortest horizon.
alert_layer = folium.FeatureGroup(name=f"ST-GNN alerts ({CFG.horizons[0]}h)", show=True)
cluster = MarkerCluster().add_to(alert_layer)
seg_lookup = {str(sid): i for i, sid in enumerate(NET["segment_id"])}
for forecast in forecasts[:1_500]:
    if forecast.horizon_hours != CFG.horizons[0] or forecast.alert_tier == "normal":
        continue
    i = seg_lookup[forecast.segment_id]
    folium.CircleMarker(
        location=[float(NET["latitude"][i]), float(NET["longitude"][i])],
        radius=4 + 6 * forecast.crash_probability,
        color=TIER_COLOURS[forecast.alert_tier],
        fill=True,
        fill_opacity=0.75,
        popup=folium.Popup(
            f"<b>{forecast.segment_id}</b><br>"
            f"tier: {forecast.alert_tier}<br>"
            f"p(crash {forecast.horizon_hours}h): {forecast.crash_probability:.3f}<br>"
            f"contagion: {forecast.spatial_contagion_index:.3f}<br>"
            f"drivers: {', '.join(forecast.contributing_factors)}",
            max_width=320,
        ),
    ).add_to(cluster)
alert_layer.add_to(fmap)

folium.LayerControl(collapsed=False).add_to(fmap)

folium_path = RENDER_DIR / f"{CFG.network_label}_risk_map_{RUN_ID}.html"
fmap.save(str(folium_path))
print(f"folium map -> {folium_path}")
fmap

In [ ]:
# Kepler.gl — WebGL rendering of the full national surface.
try:
    from keplergl import KeplerGl

    kepler_df = density_df.copy()
    kepler_df["risk_1h"] = latest_probs[:, 0]
    kepler_df["risk_6h"] = latest_probs[:, min(1, latest_probs.shape[1] - 1)]
    kepler_df["risk_24h"] = latest_probs[:, -1]
    kepler_df["alert_tier"] = [tier_for(p) for p in kepler_df["risk_1h"]]

    # Cap rows so the saved HTML stays openable; keep the head of the risk distribution.
    KEPLER_MAX_ROWS = 150_000
    if len(kepler_df) > KEPLER_MAX_ROWS:
        kepler_df = kepler_df.nlargest(KEPLER_MAX_ROWS, "risk_1h")

    kepler_config = {
        "version": "v1",
        "config": {
            "mapState": {"latitude": centre[0], "longitude": centre[1], "zoom": 6.2, "pitch": 45},
            "mapStyle": {"styleType": "light"},
        },
    }

    kepler_map = KeplerGl(height=720, data={"national_risk": kepler_df}, config=kepler_config)
    kepler_path = RENDER_DIR / f"{CFG.network_label}_kepler_{RUN_ID}.html"
    kepler_map.save_to_html(file_name=str(kepler_path))
    print(f"kepler map -> {kepler_path}  ({kepler_path.stat().st_size / 1024 ** 2:.1f} MiB)")
    print(f"rows rendered: {len(kepler_df):,}")
    print()
    print("In Kepler: add a Point layer on latitude/longitude, colour by risk_1h,")
    print("height by kde_density_per_km2, then switch to 3-D for the contagion structure.")
except ModuleNotFoundError:
    print("keplergl not installed — rerun the dependency cell.")
except Exception as exc:  # noqa: BLE001 - rendering must never fail the run
    print(f"Kepler rendering skipped: {type(exc).__name__}: {exc}")
    print("The analysis and exported artifacts above are unaffected.")

In [ ]:
# Colab's iframe sandbox blocks some inline widgets; serve the saved file instead.
from IPython.display import HTML, IFrame, display

if IN_COLAB:
    from google.colab import files as colab_files

    print("Download the interactive maps (they are self-contained HTML):")
    for path in sorted(RENDER_DIR.glob("*.html")):
        print(f"  {path.name}  ({path.stat().st_size / 1024 ** 2:.1f} MiB)")
    # Uncomment to push the files to your machine:
    # for path in sorted(RENDER_DIR.glob("*.html")):
    #     colab_files.download(str(path))
else:
    for path in sorted(RENDER_DIR.glob("*.html")):
        display(IFrame(src=str(path), width="100%", height=620))

## 14. Persist and hand back

Colab storage is ephemeral. Anything not copied to Drive, or committed, is gone when the
session ends.

In [ ]:
import shutil

PERSIST_TO_DRIVE = False   # set True after mounting Drive
DRIVE_TARGET = Path("/content/drive/MyDrive/road-risk-runs")

archive = shutil.make_archive(
    base_name=str(Path(ARTIFACT_ROOT.parent) / f"road_risk_run_{RUN_ID}"),
    format="zip",
    root_dir=str(ARTIFACT_ROOT),
)
print(f"archive : {archive}  ({Path(archive).stat().st_size / 1024 ** 2:.1f} MiB)")

if PERSIST_TO_DRIVE:
    DRIVE_TARGET.mkdir(parents=True, exist_ok=True)
    shutil.copy2(archive, DRIVE_TARGET / Path(archive).name)
    print(f"copied to {DRIVE_TARGET}")

print("\ncontents:")
for path in sorted(ARTIFACT_ROOT.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(ARTIFACT_ROOT)}  ({path.stat().st_size / 1024:,.0f} KiB)")

## 15. Handoff back to local development

The loop closes here. Anything from this session that is worth keeping goes back into the
repository as tested code — not as a notebook cell someone re-runs from memory.

**Promote to `src/road_risk_modeling/` with tests when:**

- [ ] A function here was called more than twice, or by more than one cell.
- [ ] A constant here disagrees with one in the repository (alert thresholds, bandwidths,
      feature ordering). Reconcile in the repository, not here.
- [ ] The architecture changed in a way the reference CPU engine should reflect.
- [ ] The ablation table shows a component earns its place — encode that as the default.
- [ ] A bug was found in a Layer 2 function. Fix it there with a regression test; do not
      work around it in this notebook.

**Definition of done for a run whose numbers get reported:**

- [ ] `REPO_REF` pinned to a commit SHA, not a branch.
- [ ] `IS_SYNTHETIC` is `False`.
- [ ] Layer 2 contract tests passed (§3.1) on that commit.
- [ ] The sparse/dense adjacency check passed (§6.4).
- [ ] `RUN_ABLATIONS = True` and the table is in the write-up.
- [ ] Calibration reported *after* temperature scaling, with the reliability diagram.
- [ ] Bootstrap confidence intervals included, not a bare point estimate.
- [ ] `model_card.json` archived alongside the artifacts.

**Back to Layer 1 (research notebook):** the ablation table, the calibration curve and the
planar-versus-network KDE disagreement are the three results that should drive the next
literature pass. If an ablation says the graph does not help, that is a research finding
about the network representation, not a bug to be tuned away.

---

### Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `CUDA out of memory` | batch × N × T too large | halve `batch_size`, raise `grad_accumulation` to compensate |
| Loss becomes `nan` under fp16 | gated activations overflow | set `amp_dtype = "bfloat16"`, or lower `lr` |
| PR-AUC ≈ positive rate | model learned the base rate only | check label construction; verify positives exist in the val slice |
| Val metrics beat train metrics | leakage across the split gap | confirm `GAP == max(horizons)` and the scaler was fitted on train only |
| KDE raster is all zeros | coordinates in degrees, not metres | re-run the projection cell; distances must be in metres |
| Kepler HTML is hundreds of MiB | too many rows embedded | lower `KEPLER_MAX_ROWS` |
| Session dies mid-training | Colab reclaimed the runtime | re-run from §9; the loop resumes from the last checkpoint |
| Adjacency check fails | CRS or units mismatch | compare `gaussian_sigma_m` against the reference call |